# Phase-Resolved Analysis: Ephemeris & Exposure Integration
This notebook demonstrates how to use the `PhaseEphemeris` protocol to assign phases to event data, select multiple specific phase intervals (like off-pulse regions), and strictly correct the mission exposure (livetime) for spectral fitting.

In [14]:
# Import standard libraries
import numpy as np
import astropy.units as u
from astropy.time import Time
from astropy.io import fits
from cosipy.util import fetch_wasabi_file

# Import protocol-aligned phase tools
from cosipy.spacecraftfile import SpacecraftHistory
from cosipy.phase_resolved_analysis.ephemeris import Ephemeris
from cosipy.phase_resolved_analysis.phase_assigner import PhaseAssigner
from cosipy.phase_resolved_analysis.phase_selector import PhaseSelector, apply_phase_exposure_correction

In [15]:
# Download the real DC2 orientation file
print("Downloading the spacecraft orientation file (this may take a moment)...")
ori_file = fetch_wasabi_file(
    'COSI-SMEX/DC2/Data/Orientation/20280301_3_month_with_orbital_info.ori', 
    output='20280301_3_month_with_orbital_info.ori', 
    checksum='416fcc296fc37a056a069378a2d30cb2'
)

In [16]:
# Download the DC2 Crab FITS data (Unbinned)
# The checksum ensures we don't re-download if the file is already cached locally.
data_file = fetch_wasabi_file(
    'COSI-SMEX/DC2/Data/Sources/Crab_DC2_3months_unbinned_data.fits.gz', 
    output='Crab_DC2_3months_unbinned_data.fits.gz', 
    unzip=True, 
    checksum="539e432bc9843d20396dd6a210772b6e"
)
print(f"Data ready at: {data_file}")

A file named Crab_DC2_3months_unbinned_data.fits already exists with the specified checksum (539e432bc9843d20396dd6a210772b6e). Skipping.


Data ready at: None


In [17]:
# For this tutorial, we will use parameters for the Crab pulsar.
mission_epoch = Time('2024-01-01T00:00:00', scale='utc')
crab_f0 = 29.946923 * u.Hz

# Initialize the Ephemeris
ephem = Ephemeris(f0=crab_f0, t0=mission_epoch)
print(f"Initialized Ephemeris with F0 = {ephem.f0}")

Initialized Ephemeris with F0 = 29.946923 Hz


In [18]:
# --- Assign Phases to the Data ---
raw_fits = 'Crab_DC2_3months_unbinned_data.fits'
phased_fits = 'Crab_DC2_3months_phased.fits'

print(f"Loading {raw_fits} and appending PULSE_PHASE column...")

# Initialize your assigner using the par file in your directory
assigner = PhaseAssigner('crab.par')

# Run the assignment to create the new phased file
assigner.add_phase_column(raw_fits, output_fits=phased_fits)

print(f"Phases successfully assigned! Saved to {phased_fits}")

Loading Crab_DC2_3months_unbinned_data.fits and appending PULSE_PHASE column...
Phases successfully assigned! Saved to Crab_DC2_3months_phased.fits


In [19]:
# --- Define Selection Intervals and Filter Events ---

# Let's select two background "off-pulse" regions
off_pulse_intervals = [(0.1, 0.25), (0.7, 0.85)]
print(f"Targeting Intervals: {off_pulse_intervals}")

# Initialize the Phase Selector
selector = PhaseSelector(ephemeris=ephem, intervals=off_pulse_intervals)

# Point to the file we just appended phases to
fits_filename = 'Crab_DC2_3months_phased.fits'

print(f"Attempting to filter {fits_filename}...")
filtered_events = selector.filter_events(fits_filename, output_fits='Crab_off_pulse.fits')
print(f"Successfully filtered events into new FITS file.")

Targeting Intervals: [(0.1, 0.25), (0.7, 0.85)]
Attempting to filter Crab_DC2_3months_phased.fits...
Successfully filtered events into new FITS file.


In [22]:
ori_file = '20280301_3_month_with_orbital_info.ori'

print(f"\nLoading SpacecraftHistory from {ori_file}...")
sc_history = SpacecraftHistory.open(ori_file)

# Let's look at the first few bins of the original livetime
print("\n--- BEFORE CORRECTION ---")
print(f"Original Livetime Array (first 5 bins): {sc_history._livetime_hist.contents[:5]}")
print(f"Phase Cut Total Width: 30% (0.15 + 0.15)")

# Apply the correction utility
print("\nApplying phase exposure correction...")
corrected_history = apply_phase_exposure_correction(sc_history, ephem, off_pulse_intervals)

print("\n--- AFTER CORRECTION ---")
print(f"Corrected Livetime Array (first 5 bins): {corrected_history._livetime_hist.contents[:5]}")
print("Exposure is now perfectly scaled to 30% and ready for downstream spectral extraction!")


Loading SpacecraftHistory from 20280301_3_month_with_orbital_info.ori...

--- BEFORE CORRECTION ---
Original Livetime Array (first 5 bins): [1. 1. 1. 1. 1.] s
Phase Cut Total Width: 30% (0.15 + 0.15)

Applying phase exposure correction...

--- AFTER CORRECTION ---
Corrected Livetime Array (first 5 bins): [0.3 0.3 0.3 0.3 0.3] s
Exposure is now perfectly scaled to 30% and ready for downstream spectral extraction!
